# Chapter 12: A Complete QML Workflow

Run all six printed listings in order using the PennyLane setup in Appendix IV. The final cell writes `chapter12_results.json` in the working directory and replaces it on reruns. Training uses exact ideal simulation; finite shots are evaluated separately. Split SDs and shot-batch SDs are descriptive, not confidence intervals.


In [ ]:
import json
from pathlib import Path
from time import perf_counter
from itertools import combinations
import numpy as np
import pennylane as qml
from pennylane import numpy as qnp
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (balanced_accuracy_score, roc_auc_score,
                             confusion_matrix, log_loss)

SPLIT_SEEDS = [11, 17, 23]
INIT_SEEDS = [7, 19, 31]
SHOT_BUDGETS = [128, 512, 2048, 8192]
STEPS, RATE = 80, 0.08
X, y = make_classification(
    n_samples=180, n_features=4, n_informative=3,
    n_redundant=0, n_clusters_per_class=1,
    weights=[0.5, 0.5], flip_y=0.01, class_sep=1.2,
    random_state=17,
)


def prepare_split(seed):
    """Return split indices, training-fitted scaler, and transformed arrays."""
    dev, test = train_test_split(
        np.arange(len(y)), test_size=0.2, stratify=y, random_state=seed
    )
    train, val = train_test_split(
        dev, test_size=0.25, stratify=y[dev], random_state=seed
    )
    scaler = StandardScaler().fit(X[train])
    arrays = [scaler.transform(X[idx]) for idx in (train, val, test)]
    return (train, val, test), scaler, arrays


def angles(A):
    """Map standardized features to angles using a prespecified clip of 3."""
    return np.clip(A, -3, 3) * (np.pi / 3)


print("Data shape/classes:", X.shape, np.bincount(y))


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier


def trig_features(A):
    """Return 40 local harmonic and cross-feature periodic controls."""
    a = angles(A)
    first = np.concatenate([np.sin(a), np.cos(a)], axis=1)
    local = [np.sin(a), np.cos(a), np.sin(2*a), np.cos(2*a)]
    pairs = [first[:, j] * first[:, k]
             for j, k in combinations(range(8), 2) if j % 4 != k % 4]
    return np.column_stack(local + pairs)


def build_classical_baselines(seed):
    """Return ordered validation grids; ties retain the first candidate."""
    C_values = [0.1, 1.0, 10.0]
    return {
        "majority": [DummyClassifier(strategy="most_frequent")],
        "logistic": [LogisticRegression(C=c, max_iter=2000)
                     for c in C_values],
        "tree": [DecisionTreeClassifier(max_depth=d, random_state=seed)
                 for d in [1, 2, 3]],
        "rbf": [SVC(C=c, gamma=g) for c in C_values
                for g in [0.1, 1.0, 10.0]],
        "trig": [make_pipeline(
            FunctionTransformer(trig_features),
            StandardScaler(), LogisticRegression(C=c, max_iter=2000)
        ) for c in C_values],
    }


In [ ]:
# Ascending basis order: wire 0 is the most significant bit.
BITS = ((np.arange(16)[:, None] >> np.arange(3, -1, -1)) & 1)
Z_VALUES = qnp.array(1 - 2 * BITS, requires_grad=False)


class QuantumClassifier:
    """Four-qubit RY classifier with a trainable affine sigmoid head."""

    def __init__(self, depth=2, entangle=True, frozen=False, seed=7):
        self.depth, self.entangle = depth, entangle
        self.frozen, self.seed = frozen, seed
        rng = np.random.default_rng(seed)
        self.theta0 = qnp.array(rng.normal(0, 0.3, (depth, 4)),
                                requires_grad=False)
        head = rng.normal(0, 0.1, 5)
        initial = head if frozen else np.r_[self.theta0.ravel(), head]
        self.params = qnp.array(initial, requires_grad=True)
        self.device = qml.device("default.qubit", wires=4)
        self.ledger, self.history = [], []

        @qml.qnode(self.device, interface="autograd", diff_method="backprop")
        def exact(A, theta):
            self.gates(A, theta)
            return qml.probs(wires=range(4))

        self.exact = exact

    def gates(self, A, theta):
        """Encode, rotate, then apply the ordered 0->1->2->3 CNOT chain."""
        for layer in range(self.depth):
            for wire in range(4):
                qml.RY(A[:, wire], wires=wire)
                qml.RY(theta[layer, wire], wires=wire)
            if self.entangle:
                for wire in range(3):
                    qml.CNOT(wires=[wire, wire + 1])

    def unpack(self, params):
        """Return circuit angles and the four weights plus bias."""
        theta = self.theta0 if self.frozen else params[:-5].reshape(-1, 4)
        return theta, params[-5:]

    def features(self, A, theta):
        """Return all four Z expectations from exact ordered probabilities."""
        return qnp.dot(self.exact(A, theta), Z_VALUES)

    def record(self, phase, tracker, start):
        """Append measured backend counts and elapsed phase time."""
        self.ledger.append({"phase": phase, "seconds": perf_counter()-start,
                            **tracker.totals})

    def fit(self, A, labels):
        """Train on standardized A and binary labels; store 81 loss values."""
        a = qnp.array(angles(A), requires_grad=False)
        labels = qnp.array(labels, requires_grad=False)
        start = perf_counter()
        with qml.Tracker(self.device) as tracker:
            cached = self.features(a, self.theta0) if self.frozen else None

            def objective(params):
                theta, head = self.unpack(params)
                z = cached if self.frozen else self.features(a, theta)
                logits = qnp.dot(z, head[:4]) + head[4]
                return (qnp.mean(qnp.logaddexp(0, logits) - labels*logits)
                        + 0.001*qnp.sum(head[:4]**2))

            optimizer = qml.AdamOptimizer(stepsize=RATE)
            for _ in range(STEPS):
                self.params, loss = optimizer.step_and_cost(
                    objective, self.params
                )
                self.history.append(float(loss))
            self.history.append(float(objective(self.params)))
        self.record("fit", tracker, start)
        return self

    def predict_proba(self, A, shots=None, seed=0, phase="predict"):
        """Return two class scores; optional shots sample all Zs jointly."""
        a = qnp.array(angles(A), requires_grad=False)
        theta, head = self.unpack(self.params)
        start = perf_counter()
        if shots is None:
            with qml.Tracker(self.device) as tracker:
                z = self.features(a, theta)
        else:
            device = qml.device("default.qubit", wires=4, seed=seed)

            @qml.set_shots(shots)
            @qml.qnode(device)
            def sample():
                self.gates(a, theta)
                return qml.sample(wires=range(4))

            with qml.Tracker(device) as tracker:
                z = (1 - 2*np.asarray(sample())).mean(axis=1)
        logits = np.asarray(z @ head[:4] + head[4], dtype=float)
        p = 1 / (1 + np.exp(-logits))
        self.record(phase, tracker, start)
        return np.column_stack([1-p, p])

    def resource_record(self):
        """Return logical resources, parameters, and measured phase records."""
        theta, _ = self.unpack(self.params)
        r = qml.specs(self.exact)(np.zeros((1, 4)), theta).resources
        return {"qubits": 4, "depth": r.depth, "gates": r.num_gates,
                "CNOT": r.gate_types.get("CNOT", 0),
                "stored_angles": 4*self.depth, "trained": len(self.params),
                "ledger": [dict(row) for row in self.ledger]}


In [ ]:
def metrics(labels, scores, pred):
    """Report the primary label metric, AUROC, and confusion counts."""
    return {"balanced_accuracy": float(balanced_accuracy_score(labels, pred)),
            "auroc": float(roc_auc_score(labels, scores)),
            "confusion": confusion_matrix(labels, pred).tolist()}


def classical_scores(model, A):
    """Use continuous decision scores for AUROC, never hard labels."""
    if hasattr(model, "decision_function"):
        return model.decision_function(A)
    return model.predict_proba(A)[:, 1]


VARIANTS = {"quantum": (2, True, False), "no_entanglers": (2, False, False),
            "depth_one": (1, True, False), "frozen": (2, True, True)}
records, search_records, selected_quantum, split_records = [], [], {}, []

for split_seed in SPLIT_SEEDS:
    indices, scaler, arrays = prepare_split(split_seed)
    train, val, test = indices
    A, V, T = arrays
    split_records.append({"seed": split_seed, "indices": [i.tolist() for i in indices],
                          "mean": scaler.mean_.tolist(), "scale": scaler.scale_.tolist()})
    groups = build_classical_baselines(split_seed)
    groups.update({name: [QuantumClassifier(*config, seed=s) for s in INIT_SEEDS]
                   for name, config in VARIANTS.items()})
    for name, candidates in groups.items():
        trials = []
        for index, model in enumerate(candidates):
            start = perf_counter()
            model.fit(A, y[train])
            fit_seconds = perf_counter() - start
            start = perf_counter()
            if name in VARIANTS:
                val_scores = model.predict_proba(V, phase="validation")[:, 1]
                val_pred = (val_scores >= 0.5).astype(int)
            else:
                val_pred = model.predict(V)
            val_seconds = perf_counter() - start
            score = float(balanced_accuracy_score(y[val], val_pred))
            trial = {"split": split_seed, "family": name, "candidate": index,
                     "validation": score, "fit_seconds": fit_seconds,
                     "validation_seconds": val_seconds}
            if name in VARIANTS:
                trial.update(
                    seed=model.seed, history=model.history,
                    params=np.asarray(model.params).tolist(),
                    resources=model.resource_record(),
                )
            else:
                trial["settings"] = str(model)
            trials.append(trial)
        # First maximum wins ties; no test data participate in selection.
        best = max(range(len(trials)), key=lambda j: trials[j]["validation"])
        model = candidates[best]
        start = perf_counter()
        if name in VARIANTS:
            scores = model.predict_proba(T, phase="test")[:, 1]
            pred = (scores >= 0.5).astype(int)
            selected_quantum[split_seed, name] = model
        else:
            scores = classical_scores(model, T)
            pred = model.predict(T)
        test_seconds = perf_counter() - start
        result = {"split": split_seed, "family": name, "candidate": best,
                  "validation": trials[best]["validation"],
                  "test": metrics(y[test], scores, pred),
                  "scores": scores.tolist(), "pred": pred.tolist(),
                  "search_fit_seconds": sum(t["fit_seconds"] for t in trials),
                  "search_validation_seconds": sum(t["validation_seconds"] for t in trials),
                  "test_seconds": test_seconds}
        if name in VARIANTS:
            result["resources"] = model.resource_record()
            result["test"]["log_loss"] = float(log_loss(y[test], np.c_[1-scores, scores]))
        records.append(result)
        search_records.extend(trials)
        print(split_seed, name, "val/test", round(result["validation"], 3),
              round(result["test"]["balanced_accuracy"], 3), flush=True)


In [ ]:
shot_records = []
for split_seed in SPLIT_SEEDS:
    indices, _, arrays = prepare_split(split_seed)
    T, labels = arrays[2], y[indices[2]]
    model = selected_quantum[split_seed, "quantum"]
    exact = next(r for r in records if r["split"] == split_seed
                 and r["family"] == "quantum")
    for shots in SHOT_BUDGETS:
        for repeat in range(10):
            sample_seed = 100000*split_seed + 10000*repeat + shots
            scores = model.predict_proba(T, shots, sample_seed, phase="shot_test")[:, 1]
            pred = (scores >= 0.5).astype(int)
            shot_records.append({"split": split_seed, "shots": shots,
                "repeat": repeat, "seed": sample_seed,
                "metrics": metrics(labels, scores, pred),
                "flip_rate": float(np.mean(pred != exact["pred"])),
                "scores": scores.tolist(), "pred": pred.tolist(),
                "resources": dict(model.ledger[-1])})
print("Completed shot batches:", len(shot_records))


In [ ]:
def summarize(values):
    """Return mean and descriptive sample SD, not an independence-based CI."""
    return float(np.mean(values)), float(np.std(values, ddof=1))


for name in dict.fromkeys(r["family"] for r in records):
    rows = [r for r in records if r["family"] == name]
    score = summarize([r["test"]["balanced_accuracy"] for r in rows])
    auc = summarize([r["test"]["auroc"] for r in rows])
    fit_time = sum(r["search_fit_seconds"] for r in rows)
    print(name, "BA mean/SD", score, "AUROC mean/SD", auc,
          "all fits (s)", fit_time)
for shots in SHOT_BUDGETS:
    batches = [[r for r in shot_records if r["split"] == seed
                and r["shots"] == shots] for seed in SPLIT_SEEDS]
    means = [np.mean([r["metrics"]["balanced_accuracy"] for r in b]) for b in batches]
    within = [np.std([r["metrics"]["balanced_accuracy"] for r in b], ddof=1) for b in batches]
    print("shots", shots, "split-mean BA/SD", summarize(means),
          "mean within-model SD", np.mean(within))

artifact = {"data_seed": 17, "split_seeds": SPLIT_SEEDS,
            "init_seeds": INIT_SEEDS, "steps": STEPS, "rate": RATE,
            "splits": split_records, "search": search_records,
            "results": records, "shot_sweep": shot_records}
Path("chapter12_results.json").write_text(json.dumps(artifact, indent=2))
print("Saved chapter12_results.json")
